# Семинар 12. Из выгрузки получается расчёт

В лекции мы прошли путь от строк магазина до свода и вывода. Теперь соберём несколько участков такого расчёта сами. На входе будут небольшие таблицы и массивы; читать файлы и искать данные в интернете не придётся.

Важно не только получить красивую сумму. Нужно сохранить исходные данные, заметить неподходящий ключ и различить настоящий ноль и неизвестное значение.

## Цели

- считать по массивам NumPy и понимать, что означает каждая позиция;
- отбирать строки по явным правилам, сохраняя сведения о пропусках;
- группировать записи и различать число строк и сумму;
- соединять таблицы с проверкой ключей;
- присоединять рекламу после расчёта продаж, не теряя расходы без выручки.

## Перед началом

Нужны NumPy и pandas из лекции 12. Все примеры рассчитаны на Python 3.14, NumPy 2.4+ и pandas 3.0+. Используются только переданные объекты, без файлов и сети.

В суммах храним целые копейки. Массивы в первой задаче уже имеют `dtype="int64"`. У таблиц известны столбцы; содержимое проверяем там, где это сказано в условии. Исходные массивы и таблицы менять нельзя.

Ниже оставлены заготовки с `pass` и функции проверки. После реализации вызовите соответствующую `check_task_N()`: готовые вызовы закомментированы, поэтому исходный ноутбук выполняется сверху вниз.

In [ ]:
import numpy as np
import pandas as pd


def expect_value_error(operation):
    try:
        operation()
    except ValueError:
        return
    raise AssertionError("Ожидался ValueError")

## Задание 1. Сколько осталось от каждой позиции

Реализуйте `line_margins(prices, costs, units)`. Три одномерных массива NumPy с `dtype="int64"` описывают одинаковые позиции: цену одной штуки, стоимость её закупки и количество. Функция возвращает новый массив `int64` с результатом `(цена − закупка) × количество` для каждой позиции.

Все три длины должны совпадать. Другая размерность, разные длины или любое отрицательное входное число — `ValueError`. Нули и пустые массивы допустимы. Отрицательный **результат** разрешён: товар могли продать дешевле закупки. Значения не превышают 10⁷ копеек и 1000 штук, длина — до 100 000; вычисления помещаются в `int64`.

Работайте с целыми массивами, без построчного цикла и `float`. Порядок результата совпадает с порядком входа. Это остаток после закупки товара, без рекламы и других расходов.

In [ ]:
def line_margins(prices: np.ndarray, costs: np.ndarray, units: np.ndarray) -> np.ndarray:
    pass

In [ ]:
def check_task_1():
    prices = np.array([100, 20, 0], dtype="int64")
    costs = np.array([40, 30, 0], dtype="int64")
    units = np.array([3, 1, 2], dtype="int64")
    before = [array.copy() for array in (prices, costs, units)]
    result = line_margins(prices, costs, units)
    np.testing.assert_array_equal(result, np.array([180, -10, 0], dtype="int64"))
    assert result.dtype == np.dtype("int64")
    for original, saved in zip((prices, costs, units), before):
        np.testing.assert_array_equal(original, saved)
    empty = np.array([], dtype="int64")
    assert line_margins(empty, empty, empty).shape == (0,)
    expect_value_error(lambda: line_margins(prices, costs[:1], units))
    expect_value_error(lambda: line_margins(prices.reshape(1, 3), costs, units))
    expect_value_error(lambda: line_margins(np.array([-1]), np.array([0]), np.array([1])))


# check_task_1()

## Задание 2. Пустое количество не равно нулю

В `valid_sales_rows(table)` приходят строки выгрузки. Столбцы `units` и `unit_price_kopecks` имеют строковый тип pandas; в них бывают пробелы, пропуски и неверный текст.

Верните новую таблицу только с подходящими строками: количество — конечное целое число строго больше нуля, цена — конечное целое число не меньше нуля. Пробелы по краям чисел разрешены. Пропуск, дробь, бесконечность и неверный текст делают строку неподходящей. Нулевая цена допустима: это может быть подарок.

В результате эти два столбца имеют `int64`. Остальные столбцы, их порядок и исходные метки индекса сохраняются. Ничего не подставляйте вместо плохого значения и не исправляйте исходную таблицу. Числа по модулю не превышают 10⁷; до 100 000 строк. Для разбора подойдёт `pd.to_numeric(..., errors="coerce")`, для конечности — `np.isfinite`.

In [ ]:
def valid_sales_rows(table: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
def check_task_2():
    table = pd.DataFrame({
        "event_id": list("abcdefgh"),
        "units": [" 2 ", "", "1.5", "3", None, "-1", "0", "4"],
        "unit_price_kopecks": ["100", "20", "100", "0", "20", "40", "50", "inf"],
    }, index=[9, 2, 8, 7, 3, 5, 1, 6]).astype("string")
    before = table.copy(deep=True)
    expected = pd.DataFrame({"event_id": ["a", "d"], "units": [2, 3], "unit_price_kopecks": [100, 0]}, index=[9, 7])
    expected["event_id"] = expected["event_id"].astype("string")
    pd.testing.assert_frame_equal(valid_sales_rows(table), expected)
    pd.testing.assert_frame_equal(table, before)
    empty = table.iloc[:0].copy()
    expected_empty = expected.iloc[:0].copy()
    pd.testing.assert_frame_equal(valid_sales_rows(empty), expected_empty)


# check_task_2()

## Задание 3. Каналы и итоговые суммы

В `channel_totals(events)` уже переданы корректные факты: `channel` — непустая строка, `signed_revenue_kopecks` — целая сумма. Продажа положительна, возврат отрицателен, нулевая сумма тоже допустима.

Верните таблицу со столбцами `channel`, `revenue_kopecks`, `events`. Для каждого канала нужно сложить суммы и посчитать **число исходных строк**. Это не число покупателей и не количество проданных штук.

Каналы отсортированы по возрастанию, индекс результата — новый `RangeIndex`. `channel` имеет тип `string`, оба числовых столбца — `int64`. Канал с нулевым итогом сохраняется. Пустой вход даёт пустой результат с теми же столбцами и типами. Вход уже без пропусков; до 100 000 строк, суммы и их итоги помещаются в `int64`.

In [ ]:
def channel_totals(events: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
def check_task_3():
    events = pd.DataFrame({"channel": ["сайт", "магазин", "сайт", "магазин"], "signed_revenue_kopecks": [200, 50, -100, 0]})
    events["channel"] = events["channel"].astype("string")
    before = events.copy(deep=True)
    expected = pd.DataFrame({"channel": ["магазин", "сайт"], "revenue_kopecks": [50, 100], "events": [2, 2]})
    expected["channel"] = expected["channel"].astype("string")
    pd.testing.assert_frame_equal(channel_totals(events), expected)
    pd.testing.assert_frame_equal(events, before)
    pd.testing.assert_frame_equal(channel_totals(events.iloc[:0]), expected.iloc[:0])


# check_task_3()

## Задание 4. Цена закупки должна присоединиться один раз

`add_costs(lines, products)` получает строки с `product_id`, `units` и справочник с `product_id`, `unit_cost_kopecks`. Идентификаторы — строки, ведущие нули значимы. Количества и цены уже проверены.

Присоедините стоимость закупки к каждой строке. Результат содержит ровно `product_id`, `units`, `unit_cost_kopecks`, в исходном порядке строк `lines`, с новым `RangeIndex`. Строковые ключи имеют `string`, числа — `int64`.

В справочнике не может быть повторных ключей, даже если повтор относится к неиспользованному товару. В любой таблице ключ не может быть пропущен или пуст. Если товара нет в справочнике, нужен `ValueError`: терять строку или подставлять нулевую закупку нельзя. Неизвестный ключ, повтор и пропуск — разные причины, но во всех трёх случаях проверяется тип ошибки `ValueError`.

Пустые `lines` допустимы, но справочник всё равно должен быть корректен. До 100 000 строк. Одна строка слева должна получить ровно одну запись справа; проверьте это через `merge(..., validate="many_to_one")` и проверку совпадения ключей.

In [ ]:
def add_costs(lines: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
def check_task_4():
    lines = pd.DataFrame({"product_id": ["002", "001", "002"], "units": [2, 1, 3]}, index=[7, 3, 9])
    products = pd.DataFrame({"product_id": ["001", "002"], "unit_cost_kopecks": [50, 80]}, index=[20, 10])
    lines["product_id"] = lines["product_id"].astype("string")
    products["product_id"] = products["product_id"].astype("string")
    saved_lines, saved_products = lines.copy(deep=True), products.copy(deep=True)
    expected = pd.DataFrame({"product_id": ["002", "001", "002"], "units": [2, 1, 3], "unit_cost_kopecks": [80, 50, 80]})
    expected["product_id"] = expected["product_id"].astype("string")
    pd.testing.assert_frame_equal(add_costs(lines, products), expected)
    pd.testing.assert_frame_equal(lines, saved_lines)
    pd.testing.assert_frame_equal(products, saved_products)
    expect_value_error(lambda: add_costs(lines, pd.concat([products, products.iloc[:1]], ignore_index=True)))
    expect_value_error(lambda: add_costs(lines.assign(product_id="999"), products))
    missing = lines.copy()
    missing.loc[7, "product_id"] = pd.NA
    expect_value_error(lambda: add_costs(missing, products))
    pd.testing.assert_frame_equal(add_costs(lines.iloc[:0], products), expected.iloc[:0])


# check_task_4()

## Задание 5. Реклама была, продаж не было

В `after_ads(sales, advertising)` приходят уже готовые итоги по паре `month`, `channel`. В `sales` есть ещё `revenue_kopecks`, `gross_margin_kopecks`; в `advertising` — `ad_spend_kopecks`.

Пара должна быть уникальна в каждой таблице, ключи непустые и без пропусков. Суммы продаж и маржи — `int64`, могут быть отрицательными из-за возвратов. Расход рекламы — неотрицательный `int64`; его нужно проверить. Месяцы уже записаны как `YYYY-MM`, строки имеют тип `string`.

Верните столбцы `month`, `channel`, `revenue_kopecks`, `gross_margin_kopecks`, `ad_spend_kopecks`, `after_ads_kopecks`, где последний равен марже минус реклама. Сортировка — сначала месяц, затем канал по возрастанию, индекс — `RangeIndex`, суммы — `int64`.

Группа с рекламой без продаж должна остаться: её выручка и маржа равны нулю, остаток отрицателен. Здесь отсутствие строк продаж означает, что в этой полностью проверенной выгрузке продаж не было. Если же есть продажи без строки рекламного бюджета, расход **неизвестен**: поднимите `ValueError`. Нулевая реклама должна быть записана явно.

Повтор пары, пустой ключ или отрицательная реклама — `ValueError`. Пустые таблицы допустимы с теми же правилами. До 100 000 групп, все итоги помещаются в `int64`. Исходные таблицы не меняются. Рекламу нельзя присоединять к каждой продаже: она уже относится к группе целиком. Результат после товара и рекламы не является чистой прибылью — другие расходы здесь не заданы.

При внешнем соединении в целых столбцах появляются пропуски. Чтобы не потерять точность больших сумм на промежуточном переходе к `float`, пригодится nullable-тип `Int64` (с большой `I`): он хранит целые числа вместе с `pd.NA`. На выходе снова нужен обычный `int64`, поскольку все пропуски уже объяснены или отклонены.

In [ ]:
def after_ads(sales: pd.DataFrame, advertising: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
def check_task_5():
    sales = pd.DataFrame({"month": ["2026-01", "2026-01"], "channel": ["поиск", "прямой"], "revenue_kopecks": [1000, 200], "gross_margin_kopecks": [300, 80]})
    advertising = pd.DataFrame({"month": ["2026-01"] * 3, "channel": ["поиск", "прямой", "соцсети"], "ad_spend_kopecks": [500, 0, 90]})
    for table in [sales, advertising]:
        table[["month", "channel"]] = table[["month", "channel"]].astype("string")
    saved = [table.copy(deep=True) for table in [sales, advertising]]
    expected = pd.DataFrame({"month": ["2026-01"] * 3, "channel": ["поиск", "прямой", "соцсети"], "revenue_kopecks": [1000, 200, 0], "gross_margin_kopecks": [300, 80, 0], "ad_spend_kopecks": [500, 0, 90], "after_ads_kopecks": [-200, 80, -90]})
    expected[["month", "channel"]] = expected[["month", "channel"]].astype("string")
    pd.testing.assert_frame_equal(after_ads(sales, advertising), expected)
    for table, before in zip([sales, advertising], saved):
        pd.testing.assert_frame_equal(table, before)
    expect_value_error(lambda: after_ads(sales, advertising.iloc[1:]))
    expect_value_error(lambda: after_ads(sales, pd.concat([advertising, advertising.iloc[:1]], ignore_index=True)))
    expect_value_error(lambda: after_ads(sales, advertising.assign(ad_spend_kopecks=-1)))
    large = sales.iloc[:1].copy()
    large["revenue_kopecks"] = 2**53 + 1
    large["gross_margin_kopecks"] = 2**53 + 1
    assert after_ads(large, advertising).loc[0, "after_ads_kopecks"] == 2**53 + 1 - 500
    pd.testing.assert_frame_equal(after_ads(sales.iloc[:0], advertising.iloc[:0]), expected.iloc[:0])


# check_task_5()

## Итоги

У каждой части расчёта свой вопрос. Массив считает одинаковую формулу по позициям. Маска отбирает допустимые строки. Группировка собирает факты нужного уровня. Соединение добавляет сведения, не меняя числа событий. Последний расчёт учитывает расходы, которые легко потерять при неудачном соединении.

Перед тем как объяснять результат, проверьте три вещи: что означает одна строка, какие строки не попали в расчёт и какие расходы в нём вообще учтены. Затем переходите к [домашним задачам](tasks.md).